In [2]:
import faiss
import pickle
import numpy as np
from pathlib import Path
from sentence_transformers import SentenceTransformer

# =========================
# CONFIG
# =========================
VECTOR_STORE = Path(
    r"C:\Users\kgathola.puka\OneDrive - MSC\Documents\GitHub\RCP(test)\SPEED CHATBOT PROJECT\DATA\vector_store"
)

EMBEDDING_MODEL = "all-mpnet-base-v2"
TOP_K = 8

# =========================
# LOAD VECTOR STORE
# =========================
index = faiss.read_index(str(VECTOR_STORE / "faiss.index"))

with open(VECTOR_STORE / "metadata.pkl", "rb") as f:
    chunks = pickle.load(f)

print(f"FAISS vectors   : {index.ntotal}")
print(f"Metadata chunks : {len(chunks)}")

assert index.ntotal == len(chunks), "❌ Index / metadata mismatch!"

# =========================
# LOAD MODEL
# =========================
model = SentenceTransformer(EMBEDDING_MODEL)

# =========================
# RETRIEVER
# =========================
def retrieve_chunks(query: str, top_k: int = TOP_K):
    # Encode + normalize (COSINE)
    query_vec = model.encode([query], convert_to_numpy=True)
    query_vec = query_vec / np.linalg.norm(query_vec, axis=1, keepdims=True)

    distances, indices = index.search(query_vec, top_k)

    results = []

    for rank, (idx, score) in enumerate(zip(indices[0], distances[0]), start=1):
        if idx == -1:
            continue

        meta = chunks[idx]

        results.append({
            "rank": rank,
            "score": float(score),   # cosine similarity
            "source": meta.get("source"),
            "folder": meta.get("folder"),  # 👈 IMPORTANT
            "chunk_id": meta.get("chunk_id"),
            "text": meta.get("text"),
        })

    return results


c:\Users\kgathola.puka\OneDrive - MSC\Documents\GitHub\RCP(test)\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\kgathola.puka\OneDrive - MSC\Documents\GitHub\RCP(test)\.venv\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):



FAISS vectors   : 208
Metadata chunks : 208


In [1]:
question = "How to amend GRN numbers on receipts inbound & closed on WMS for Jinko Solar"

results = retrieve_chunks(question)

for r in results:
    print("\n==============================")
    print(f"Rank   : {r['rank']}")
    print(f"Score  : {r['score']:.4f}")
    print(f"Folder : {r['folder']}")
    print(f"Source : {r['source']}")
    print(r["text"][:700])


Unexpected exception formatting exception. Falling back to standard exception


Traceback (most recent call last):
  File "c:\Users\kgathola.puka\OneDrive - MSC\Documents\GitHub\RCP(test)\.venv\Lib\site-packages\IPython\core\interactiveshell.py", line 3577, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
    ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\kgathola.puka\AppData\Local\Temp\ipykernel_32716\1783537631.py", line 3, in <module>
    results = retrieve_chunks(question)
              ^^^^^^^^^^^^^^^
NameError: name 'retrieve_chunks' is not defined

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "c:\Users\kgathola.puka\OneDrive - MSC\Documents\GitHub\RCP(test)\.venv\Lib\site-packages\IPython\core\interactiveshell.py", line 2168, in showtraceback
    stb = self.InteractiveTB.structured_traceback(
        etype, value, tb, tb_offset=tb_offset
    )
  File "c:\Users\kgathola.puka\OneDrive - MSC\Documents\GitHub\RCP(test)\.venv\Lib\site-packages\IPython\core\ult

In [27]:
from rank_bm25 import BM25Okapi
import re

def tokenize(text):
    return re.findall(r"\b\w+\b", text.lower())

corpus = [tokenize(c["text"]) for c in chunks]
bm25 = BM25Okapi(corpus)


In [28]:
def hybrid_retrieve(query, top_k=8, alpha=0.6):
    """
    alpha → weight for vector search (0–1)
    """

    # ----- VECTOR SEARCH -----
    vector_results = retrieve_chunks(query, top_k=top_k * 2)
    vector_scores = {r["chunk_id"]: r["score"] for r in vector_results}

    # Normalize vector scores
    if vector_scores:
        max_v = max(vector_scores.values())
        vector_scores = {k: v / max_v for k, v in vector_scores.items()}

    # ----- BM25 SEARCH -----
    tokenized_query = tokenize(query)
    bm25_scores = bm25.get_scores(tokenized_query)

    bm25_norm = bm25_scores / bm25_scores.max() if bm25_scores.max() > 0 else bm25_scores

    # ----- FUSION -----
    fused = []

    for idx, meta in enumerate(chunks):
        chunk_id = meta["chunk_id"]

        score = (
            alpha * vector_scores.get(chunk_id, 0)
            + (1 - alpha) * bm25_norm[idx]
        )

        if score > 0:
            fused.append({
                "chunk_id": chunk_id,
                "score": float(score),
                "source": meta["source"],
                "folder": meta.get("folder"),
                "text": meta["text"]
            })

    fused = sorted(fused, key=lambda x: x["score"], reverse=True)[:top_k]

    return fused


In [29]:
results = hybrid_retrieve(
    "How to amend GRN numbers on receipts inbound & closed on WMS for Jinko Solar"
)

for r in results:
    print("\n==============================")
    print(f"Score  : {r['score']:.4f}")
    print(f"Folder : {r['folder']}")
    print(f"Source : {r['source']}")
    print(r["text"][:700])



Score  : 1.0000
Folder : None
Source : Receipt Creation, Line Management & SQL Validation in Speed WMS.txt
8. Key RAG Questions This SOP Answers
• How do I create a receipt in Speed WMS?
• Where are receipt headers and lines stored?
• Why doesn’t stock appear after receipt creation?
• How do I validate receipts using SQL?
Sensitivity: Internal

• How do I reopen or correct a GRN?
Sensitivity: Internal

Score  : 0.8334
Folder : None
Source : Reserves GRN and SQL VALIDATION.txt
6. Reversal and Correction Scenarios
6.1 Reopening a Closed GRN
Use this SQL to reopen a closed receipt and reset Pegasus integration:
UPDATE REE_DAT
SET REE_ETRE = '100',
REE_TOP1 = 0
WHERE ACT_CODE = 'Activity Code'
AND REE_NORE = 'Receipt Number'
AND REE_KEYU = 'Receipt ID'
AND REE_CCLI = 'Customer Account Code';
7. Troubleshooting Mapping (RAG-Friendly)
• Receipt exists but no stock
Sensitivity: Internal

o Cause: Stock generation not executed
o Table to check: REL_DAT
• Stock exists but no weight
o Cause: St

#### CONTEXT WINDOW ASSEMBLY ####

In [30]:
def estimate_tokens(text: str) -> int:
    # Rough rule: 1 token ≈ 4 characters
    return len(text) // 4


In [31]:
import hashlib

def text_hash(text: str) -> str:
    return hashlib.md5(text.strip().encode("utf-8")).hexdigest()


In [32]:
def build_context(
    retrieved_chunks,
    max_tokens=1400,
    max_chunks=6
):
    context_blocks = []
    used_tokens = 0
    used_hashes = set()

    for chunk in retrieved_chunks:
        text = chunk["text"].strip()
        h = text_hash(text)

        if h in used_hashes:
            continue

        tokens = estimate_tokens(text)

        if used_tokens + tokens > max_tokens:
            break

        block = f"""
SOURCE: {chunk['source']}
FOLDER: {chunk.get('folder')}
RELEVANCE SCORE: {chunk['score']:.4f}

{text}
""".strip()

        context_blocks.append(block)
        used_tokens += tokens
        used_hashes.add(h)

        if len(context_blocks) >= max_chunks:
            break

    final_context = "\n\n---\n\n".join(context_blocks)

    return final_context


In [33]:
hybrid_results = hybrid_retrieve(
    "List all column in the receipt table and their descriptions",
    top_k=10
)

context = build_context(hybrid_results)

print(context[:2000])  # preview


SOURCE: Customization of the screens.txt
FOLDER: None
RELEVANCE SCORE: 1.0000

It is thus possible to customize the titles and adapt the layout for each element
displayed.
The list of tables in which it is possible to choose the different information differs
according to the functionality being configured.
It is possible to filter the search for items by entering a keyword in the Filter field. For
each table, the list of elements containing the entered term appears, facilitating the
search:
➢Adding an element to the current view
Sensitivity: Internal

A click on after the selection of the line to add to the table, enable to add
its column to the table.
A double-click on the element in the left list also enables to add it to the view current
view.
The element is added, by default, in the very last position.
➢Deleting an element from the current view
Selection of the column corresponding to the element to be deleted in the upper right
part, and click on to delete the information from the

#### Define the SYSTEM PROMPT ####

In [34]:
SYSTEM_PROMPT = """
You are a WMS and SQL support assistant.

RULES:
- Use ONLY the information provided in the CONTEXT.
- If the answer is not explicitly stated in the context, say:
  "This information is not available in the provided documents."
- Do NOT use prior knowledge.
- Do NOT guess.
- Be precise and procedural.
- Cite sources at the end of each section using:
  [SOURCE: filename]

FORMAT:
- Clear steps
- Bullet points where applicable
- Supporting tone
"""


In [35]:
def build_prompt(context: str, question: str) -> str:
    return f"""
CONTEXT:
{context}

QUESTION:
{question}

ANSWER:
"""


In [38]:
from groq import Groq

client = Groq(api_key="gsk_REDACTED_KEY_WAS_ROTATED")

def generate_answer(context: str, question: str):
    prompt = build_prompt(context, question)

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt}
        ],
        temperature=0.1,   
        max_tokens=1000
    )

    return response.choices[0].message.content


In [ ]:
question = "what cq "

hybrid_results = hybrid_retrieve(question, top_k=10)

context = build_context(hybrid_results)

answer = generate_answer(context, question)

print(answer)


The movement table is MVT_DAT. 

This information is available in the following sources:
* Reserves GRN and SQL VALIDATION.txt
* Receipt Creation, Line Management & SQL Validation in Speed WMS.txt
* Introduction.txt 

[SOURCE: Reserves GRN and SQL VALIDATION.txt, Receipt Creation, Line Management & SQL Validation in Speed WMS.txt, Introduction.txt]
